In [1]:
import pandas as pd
import numpy as np
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/clean_transactions.csv', parse_dates=['InvoiceDate'])
print(df.shape)
print(df.head(3))

(397885, 9)
   Invoice StockCode                         Description  Quantity  \
0   536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6   
1   536365     71053                 WHITE METAL LANTERN         6   
2   536365    84406B      CREAM CUPID HEARTS COAT HANGER         8   

          InvoiceDate  Price  Customer ID         Country  Revenue  
0 2010-12-01 08:26:00   2.55        17850  United Kingdom    15.30  
1 2010-12-01 08:26:00   3.39        17850  United Kingdom    20.34  
2 2010-12-01 08:26:00   2.75        17850  United Kingdom    22.00  


In [2]:
summary = summary_data_from_transaction_data(
    df,
    customer_id_col='Customer ID',
    datetime_col='InvoiceDate',
    monetary_value_col='Revenue',
    observation_period_end='2011-12-10'
)

print(summary.shape)
print(summary.head(10))
print(summary.describe())

(4338, 4)
             frequency  recency      T  monetary_value
Customer ID                                           
12346              0.0      0.0  326.0        0.000000
12347              6.0    365.0  368.0      599.701667
12348              3.0    283.0  359.0      301.480000
12349              0.0      0.0   19.0        0.000000
12350              0.0      0.0  311.0        0.000000
12352              6.0    260.0  297.0      368.256667
12353              0.0      0.0  205.0        0.000000
12354              0.0      0.0  233.0        0.000000
12355              0.0      0.0  215.0        0.000000
12356              2.0    303.0  326.0      269.905000
         frequency      recency            T  monetary_value
count  4338.000000  4338.000000  4338.000000     4338.000000
mean      2.864223   130.771554   223.831028      307.030232
std       5.949000   132.210509   117.854570     2612.749786
min       0.000000     0.000000     1.000000        0.000000
25%       0.000000     0.

In [3]:
returning_customers = summary[summary['frequency'] > 0]
print(f"Returning customers: {returning_customers.shape[0]}")
print(f"One-time customers excluded: {summary.shape[0] - returning_customers.shape[0]}")

Returning customers: 2790
One-time customers excluded: 1548


In [4]:
bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(
    summary['frequency'],
    summary['recency'],
    summary['T']
)
print(bgf)
print("\nModel parameters:")
print(bgf.params_)

<lifetimes.BetaGeoFitter: fitted with 4338 subjects, a: 0.00, alpha: 69.23, b: 6.65, r: 0.83>

Model parameters:
r         0.825658
alpha    69.231430
a         0.003922
b         6.653119
dtype: float64


In [5]:
t = 90
summary['predicted_purchases_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t,
    summary['frequency'],
    summary['recency'],
    summary['T']
)

print(summary[['frequency', 'recency', 'predicted_purchases_90d']].sort_values('predicted_purchases_90d', ascending=False).head(10))

             frequency  recency  predicted_purchases_90d
Customer ID                                             
14911            131.0    372.0                26.756689
12748            112.0    373.0                22.900507
17841            111.0    372.0                22.697248
15311             89.0    373.0                18.232061
14606             88.0    372.0                18.028880
12971             70.0    369.0                14.407470
13089             65.0    367.0                13.481965
14527             53.0    367.0                11.024155
13798             52.0    371.0                10.746086
16422             47.0    352.0                 9.790865


In [6]:
ggf = GammaGammaFitter(penalizer_coef=0.0)
ggf.fit(
    returning_customers['frequency'],
    returning_customers['monetary_value']
)
print(ggf)
print("\nModel parameters:")
print(ggf.params_)

<lifetimes.GammaGammaFitter: fitted with 2790 subjects, p: 2.10, q: 3.45, v: 485.92>

Model parameters:
p      2.103382
q      3.451306
v    485.915505
dtype: float64


In [7]:
clv = ggf.customer_lifetime_value(
    bgf,
    returning_customers['frequency'],
    returning_customers['recency'],
    returning_customers['T'],
    returning_customers['monetary_value'],
    time=12,
    freq='D',
    discount_rate=0.01
)

clv = clv.reset_index()
clv.columns = ['Customer ID', 'CLV']
print(clv.shape)
print(clv.describe())
print(clv.sort_values('CLV', ascending=False).head(10))

(2790, 2)
        Customer ID            CLV
count   2790.000000    2790.000000
mean   15281.868459    2699.706562
std     1717.476402    8757.638095
min    12347.000000     322.086978
25%    13812.250000     832.534002
50%    15240.500000    1392.184727
75%    16778.750000    2398.154647
max    18287.000000  221382.896465
      Customer ID            CLV
1111        14646  221382.896465
2706        18102  178308.224320
1938        16446  174510.500978
2413        17450  146990.780222
834         14096  126538.629566
1243        14911  109092.040019
34          12415   95943.588062
864         14156   89119.227990
2441        17511   67441.496455
1745        16029   58505.197261


In [8]:
clv.to_csv('../data/processed/clv_scores.csv', index=False)
print("CLV scores saved")

CLV scores saved
